# Datencheck

Umsatzprognose und Gewinn/Verlust im Überblick. Dieses Notebook liest nur, es
verändert nichts.

In [ ]:
# @title Umgebung einrichten und Dashboard laden

import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py

from datetime import date

import setup

stichtag = date.today()
horizont_monate = 3
auslastung_monate = 12
gewinn_verlust_monate = 11

dashboard = setup.dashboard(
    stichtag=stichtag,
    horizont_monate=horizont_monate,
    auslastung_monate=auslastung_monate,
)

print(dashboard.ladebericht())

## Parameter und Variablen für die Verarbeitung

In [ ]:
# @title Verbrauchsplan-Übersteuerung (optional)

# Projektname wie in 01_dashboard.ipynb (dort in der Hinweistabelle sichtbar),
# Zielmonat als (Jahr, Monat). Beispiel:
# verbrauchsplan_uebersteuerungen = {"Beispielprojekt": (2026, 12)}
verbrauchsplan_uebersteuerungen: dict[str, tuple[int, int]] = {}

dashboard.verbrauchsplan_uebersteuern(verbrauchsplan_uebersteuerungen)

In [ ]:
# @title Interner-Arbeit-Abschlag für die Simulation (optional)

# Abschlag auf die verfügbare Kapazität, um den Anteil interner Arbeit in der
# Simulation zu berücksichtigen (0.0 = keine Reduktion, wie bisher). Vorbelegt mit
# dem historischen Durchschnitt (Grafik/Tabelle dazu in notebooks/01_dashboard.ipynb),
# per Hand änderbar.
interne_arbeit_abschlag = dashboard.durchschnittlicher_anteil_interner_arbeit() or 0.0

In [ ]:
# @title Simulation ausführen

setup.simulieren(dashboard, monate=horizont_monate, interne_arbeit_abschlag=interne_arbeit_abschlag)

## Gewinn/Verlust je Monat

Umsatz minus Kosten für die letzten `gewinn_verlust_monate` abgeschlossenen Monate,
anschließend die Vorausschau über den mit `horizont_monate` konfigurierten
Prognosehorizont (gedämpfte Balken) - dieselbe Simulation wie in der Umsatzprognose.

In [ ]:
# @title Gewinn/Verlust je Monat

dashboard.gewinn_verlust_monatlich(monate=gewinn_verlust_monate)

## Gewinn/Verlust je Jahr

Dieselbe Kennzahl wie oben, aber über die gesamte geladene Historie (alle in
`KOSTEN_SHEET_IDS` konfigurierten Jahre) und je Kalenderjahr eine eigene Linie auf
gemeinsamer Monatsachse - ein Jahresvergleich statt eines fortlaufenden Zeitstrahls.
Die gestrichelte Fortsetzung des jüngsten Jahres ist die Vorausschau über den
Prognosehorizont.


In [ ]:
# @title Gewinn/Verlust je Jahr

dashboard.gewinn_verlust_je_jahr()

## Kumulierte Umsatzrendite je Jahr

Gewinn geteilt durch Umsatz, kumuliert von Januar bis zum jeweiligen Monat (eine
Year-to-Date-Marge, keine Summe einzelner Monatsprozente) - je Kalenderjahr eine
eigene Linie, mit derselben Vorausschau-Konvention wie oben.


In [ ]:
# @title Kumulierte Umsatzrendite

dashboard.umsatzrendite_kumuliert()